# Boundless Prover Profitability Model

> Source: https://github.com/zerokrab/boundless-jupyter

**Core objective:** Assess the profitability of running a Boundless Prover under different ZKC prices, market utilization rates, and GPU configurations.

## Model Overview

To estimate profitability we use several inputs:
- GPU configurations - to adjust costs and MHz capacity
- ZKC prices - to model for different price points
- Market reward rates - to model different expected USD per MHz from market orders
- Market reward rate - average USD per MHz expected from the market

All costs, revenue, and profit are expressed per epoch and primarily denominated in USD.

## How to run

### Online
1. (Optional) Fetch the latest epoch data:
    - Goto https://explorer.boundless.network/epochs
    - In the top right click "Export CSV"
    - Upload the file to Jupyter, overwriting the existing copy
1. In the top menu, select "Run" -> "Run All Cells"

### Local
1. Install dependencies: `pip install -r requirements.txt`
2. Edit the **1.1 Model Parameters** section to adjust the model parameters
3. (Optional) Fetch the latest epoch data:
    - Goto https://explorer.boundless.network/epochs
    - In the top right click "Export CSV"
    - Replace `epochs.csv` with the new copy
3. To run this locally: `jupyter notebook` will launch the web interface. Google Collab can also be used to run this online.

## 1. Inputs

This section defines the core setup for the model, split into three parts:

1. **Model Parameters**: Tunable inputs for the model. (EDIT THESE).
2. **Constants**: Fixed values required for calculation (DO NOT EDIT).
3. **POVW Reward Data**: Parsing logic that derives `POVW_ZKC_PER_MHZ_PER_EPOCH` from historical epochs data (epochs.csv).

### 1.1 Model Parameters (EDIT THESE)

`ZKC_PRICES_USD`

The model will use each value as a possible scenario for the price of ZKC. Pick any values you want.

`MARKET_ORDER_UTIL`

A fixed percentage of GPU capacity used for market orders. (Note: utilization tends to be <50%)

`MARKET_REWARD_USD_PER_MHZ`

A range of expected prices paid per million cycles on the market. The model will use each value as a scenario. Can be estimated from [market stats](https://explorer.boundless.network/stats), (see "Lock price per cycle distribution"). You can divide USD/Bcycle by 1000 to get USD/Mhz.

`FIXED_COST_MONTHLY_USD`

Any additional expenses, fixed cost per month.

`gpu_configs`

List of GPU options to model.

In [ ]:
import pandas as pd
import numpy as np
import re

# ZKC prices (USD per ZKC) to model
ZKC_PRICES_USD = [0.025, 0.05, 0.075, 0.1, 0.125, 0.15, 0.2, 0.3, 0.5, 1.0]

# Market order utilization: fixed percentage of capacity used for market orders
MARKET_ORDER_UTIL = 0.5

# Average reward per million cycles in USD (range of scenarios)
MARKET_REWARD_USD_PER_MHZ = [0.00003, 0.00005, 0.00007, 0.0001]

# Fixed cost additional expenses
FIXED_COST_MONTHLY_USD = 0

# GPU configurations
gpu_configs = pd.DataFrame([
    {"label": "RTX5090 x 8", "num_gpus": 8, "usd_per_hour": 0.50, "mhz": 1.1},
    {"label": "RTX5090 x 4", "num_gpus": 4, "usd_per_hour": 0.55, "mhz": 1.1},
    {"label": "RTX4090 x 8", "num_gpus": 8, "usd_per_hour": 0.40, "mhz": 0.9},
])

### 1.2 Constants (DO NOT EDIT)

In [ ]:
# Epoch: 48 hours. ~15 epochs per 30-day month.
HOURS_PER_EPOCH = 48
SECONDS_PER_EPOCH = HOURS_PER_EPOCH * 3600  # 172800
EPOCHS_PER_MONTH = 30 * 24 / HOURS_PER_EPOCH  # 15

In [ ]:
# Additional Calculations (DO NOT EDIT)
fixed_cost_per_epoch = FIXED_COST_MONTHLY_USD / EPOCHS_PER_MONTH
gpu_configs["rental_per_epoch_usd"] = gpu_configs["usd_per_hour"] * gpu_configs["num_gpus"] * HOURS_PER_EPOCH
# Million cycles per epoch = mhz per GPU (per sec) * num_gpus * seconds per epoch
gpu_configs["mhz_per_epoch"] = gpu_configs["mhz"] * gpu_configs["num_gpus"] * SECONDS_PER_EPOCH

gpu_configs["cost_per_mhz_usd"] = gpu_configs["rental_per_epoch_usd"] / gpu_configs["mhz_per_epoch"]

gpu_configs["total_cost_epoch"] = gpu_configs["rental_per_epoch_usd"] + fixed_cost_per_epoch

cost_summary = gpu_configs[
    ["label", "num_gpus", "rental_per_epoch_usd", "total_cost_epoch", "cost_per_mhz_usd"]
].copy()
cost_summary

### 1.3 POVW Reward Data

Using historical data (epochs.csv) we compute the average (mean) ZKC reward per million cycles.

We ignore the latest epoch as it is ongoing. `EPOCH_LOOKBACK_COUNT` can be adjusted to set the number of most recent epochs used in the calculation.

In [ ]:
EPOCH_LOOKBACK_COUNT=10

def parse_cycles(s):
    """Parse Total Cycles string (e.g. '664T', '791M', '0') to cycle count."""
    s = str(s).strip().upper().replace(",", "")
    if not s or s == "0":
        return 0.0
    m = re.match(r"^([\d.]+)\s*([TGMK]?)$", s)
    if not m:
        return np.nan
    val = float(m.group(1))
    suffix = m.group(2) or ""
    mult = {"T": 1e12, "G": 1e9, "M": 1e6, "K": 1e3}.get(suffix, 1)
    return val * mult


def parse_zkc(s):
    """Parse Mining Rewards string (e.g. '288K ZKC', '96,011 ZKC') to ZKC amount."""
    s = str(s).strip().replace(",", "").replace(" ZKC", "").upper()
    if not s or s == "0":
        return 0.0
    m = re.match(r"^([\d.]+)\s*([KMB]?)$", s)
    if not m:
        return np.nan
    val = float(m.group(1))
    suffix = m.group(2) or ""
    mult = {"B": 1e9, "M": 1e6, "K": 1e3}.get(suffix, 1)
    return val * mult


epochs = pd.read_csv("epochs.csv")
# Exclude latest epoch (first row): still ongoing, incomplete.
epochs_complete = epochs.iloc[1:].copy()
# Parse units
epochs_complete["total_cycles"] = epochs_complete["Total Cycles"].map(parse_cycles)
epochs_complete["mining_rewards_zkc"] = epochs_complete["Mining Rewards (ZKC)"].map(parse_zkc)
# Convert cycles => mhz
epochs_complete["mhz"] = epochs_complete["total_cycles"] / 1e6  # million cycles
# Compute zkc_per_mhz
epochs_complete["zkc_per_mhz"] = np.where(
    epochs_complete["mhz"] > 0,
    epochs_complete["mining_rewards_zkc"] / epochs_complete["mhz"],
    np.nan,
)

# Use only EPOCH_LOOKBACK_COUNT most recent completed epochs to avoid older values skewing the mean
epochs_for_povw = epochs_complete.head(EPOCH_LOOKBACK_COUNT)
POVW_ZKC_PER_MHZ_PER_EPOCH = epochs_for_povw["zkc_per_mhz"].mean()

# Display table showing parsed data for each epoch
epochs_complete[["Epoch", "mining_rewards_zkc", "mhz", "zkc_per_mhz"]].rename(
    columns={"mining_rewards_zkc": "ZKC"}
)

## 2. Compute Cost, Revenue, Profit

**Profit per epoch** = total revenue − total cost (GPU + fixed).

In [ ]:
results = []
for _, row in gpu_configs.iterrows():
    mhz_per_epoch = row["mhz_per_epoch"]
    total_cost_epoch = row["total_cost_epoch"]
    scenario = row["label"]
    for zkc_price in ZKC_PRICES_USD:
        povw_revenue = mhz_per_epoch * POVW_ZKC_PER_MHZ_PER_EPOCH * zkc_price
        for market_reward in MARKET_REWARD_USD_PER_MHZ:
            market_revenue = mhz_per_epoch * market_reward * MARKET_ORDER_UTIL
            total_revenue = povw_revenue + market_revenue
            profit = total_revenue - total_cost_epoch
            results.append({
                "scenario": scenario,
                "label": row["label"],
                "num_gpus": row["num_gpus"],
                "zkc_price_usd": zkc_price,
                "market_order_util": MARKET_ORDER_UTIL,
                "market_reward_usd_per_mhz": market_reward,
                "mhz_per_epoch": mhz_per_epoch,
                "cost_per_epoch": total_cost_epoch,
                "povw_revenue": povw_revenue,
                "market_revenue": market_revenue,
                "total_revenue": total_revenue,
                "profit_per_epoch": profit,
            })

revenue_df = pd.DataFrame(results)
# Display all data
revenue_df

In [ ]:
# Export results for standalone dashboard
revenue_df.to_csv("results.csv", index=False)

In [ ]:
# Display summary of data
revenue_df[["scenario", "zkc_price_usd", "market_reward_usd_per_mhz", "profit_per_epoch"]]

## 3. Visualize results

### 3.1 Raw Results

In [ ]:
profit_pivot = revenue_df.pivot_table(
    index=["scenario", "market_reward_usd_per_mhz"], columns="zkc_price_usd", values="profit_per_epoch"
)
profit_pivot

### 3.2 Scenario Graphs

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# One graph per GPU config; each shows profit vs ZKC price with one line per market reward rate
for scenario in revenue_df["scenario"].unique():
    fig, ax = plt.subplots(figsize=(10, 4))
    scenario_df = revenue_df[revenue_df["scenario"] == scenario]
    for reward, sub in scenario_df.groupby("market_reward_usd_per_mhz"):
        ax.plot(sub["zkc_price_usd"], sub["profit_per_epoch"], marker="o", label=f"${reward:.5f}/MHz")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_xlabel("ZKC price (USD)")
    ax.set_ylabel("Profit per epoch (USD)")
    ax.set_title(f"{scenario}: profit per epoch vs ZKC price")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 3.3 Profitable scenarios (profit_per_epoch > 0)

Pivot table filtered to only show scenarios with positive `profit_per_epoch`.

In [ ]:
# Pivot table: profitable scenarios only (profit_per_epoch > 0)
profitable_revenue_df = revenue_df[revenue_df["profit_per_epoch"] > 0].copy()

if profitable_revenue_df.empty:
    print("No profitable scenarios for current inputs (profit_per_epoch <= 0 everywhere).")
else:
    profitable_pivot = profitable_revenue_df.pivot_table(
        index=["scenario", "market_reward_usd_per_mhz"],
        columns="zkc_price_usd",
        values="profit_per_epoch",
    )
    profitable_pivot

## 4. Interactive Dashboard (Panel)


In [ ]:
# Install Panel (no-op if already installed)
%pip install -q panel

In [ ]:
from dashboard import build_dashboard
build_dashboard(revenue_df)